# Week 7 capstone: Step 6, the main experiment (3 arms x 3 seeds, frozen protocol)

**This is the frozen nine-run experiment.** It runs arms A (learned absolute), B (sinusoidal) and C (RoPE) on the same seeds 0, 1 and 2, 8000 steps, policy P1, the existing dataset, in **one session on one GPU**, on the frozen commit. Arm A is run again here under the same protocol as B and C; the pilot's arm A results are not used.

- **Order.** Seed blocks with the arm order rotated by seed (seed 0: A,B,C; seed 1: B,C,A; seed 2: C,A,B). Each run takes about 5-6 minutes, so the whole experiment takes roughly **50-60 minutes**.
- **Reported for every run:** validation MAPE, training time, batch-1 inference latency, peak GPU memory. Every attempt (completed, failed, interrupted) is kept.
- **The decision rule is frozen** (δ = 0.10, k = 2). At the end the notebook prints the matched differences and the frozen rule's verdict. These are exploratory screening criteria for a three-seed study, not significance tests. Do not interpret them yet; send the zip back for review.
- **No `pip install`, no upgrades, no optional seeds, no pilot or calibration cells.** The extension is a separate stage that the repository refuses without the user's approval.

**Rules for this session**
1. Use a **T4 GPU** runtime and keep this browser tab open and active until the last cell finishes. A dropped session splits the experiment across sessions.
2. Run the cells top to bottom, **once**. **Do not re-run cell 1 after cell 4 has started**: it would discard the runs so far.
3. If anything stops or errors, **paste the whole output** and do not start over. A re-run resumes, but a resumed experiment has to be reported as resumed.

In [ ]:
# 1. Get the code. COMMIT is already filled in (the frozen commit), so there is nothing to edit.
REPO_URL = "https://github.com/sweiss1367/wk5_architecture-variants.git"
BRANCH = "claude/trusting-pascal-1qjsp0"
COMMIT = "8c561c196589cc4c7ee7789dced7c95723e805c1"   # the frozen commit: the pre-registration freeze record is part of it

import os, subprocess, time
if os.path.isdir("repo"):                  # never reuse a stale copy: keep the old one aside
    os.rename("repo", f"repo_old_{int(time.time())}")
used_zip = os.path.exists("week7_repo.zip")
if used_zip:
    subprocess.run(["unzip", "-q", "week7_repo.zip"], check=True)
else:
    url = REPO_URL
    try:                                   # private repo: add a Colab secret named GITHUB_TOKEN (never printed)
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
        if tok:
            url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--branch", BRANCH, url, "repo"], check=True)
    subprocess.run(["git", "-C", "repo", "remote", "set-url", "origin", REPO_URL], check=True)
subprocess.run(["git", "-C", "repo", "checkout", COMMIT], check=True)
head = subprocess.run(["git", "-C", "repo", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code source:", "uploaded zip" if used_zip else "GitHub clone", "| HEAD:", head)
if not head.startswith(COMMIT):
    raise SystemExit(f"Wrong commit: HEAD is {head}, expected {COMMIT}. Stopping so the provenance record stays exact.")
%cd repo/week7_capstone
!git status --short

In [ ]:
# 2. GPU check. All earlier sessions used a Tesla T4. The main experiment must run on ONE GPU, so note which one this is.
!nvidia-smi --query-gpu=name,uuid,driver_version --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a T4 GPU and restart."
name = torch.cuda.get_device_name(0)
print(torch.__version__, torch.version.cuda, name)
if "T4" not in name:
    print("WARNING: not a T4. The experiment will still run, but tell the reviewer before continuing.")

In [ ]:
# 3. Gates, checked before any run: environment equals the lock, code equals the frozen commit, the freeze record holds
#    (every frozen file matches its recorded hash), policy P1 and 8000 steps as approved, and no main run exists yet.
import glob, json, subprocess, yaml
subprocess.run(["python", "src/check_env.py", "--verify"], check=True)
subprocess.run(["python", "src/freeze.py", "--verify"], check=True)
subprocess.run(["git", "diff", "--quiet", "--", ".", ":(exclude)results", ":(exclude)data"], check=True)
cfg = yaml.safe_load(open("configs/base.yaml"))
decision = json.load(open("results/determinism_decision.json"))["selected_policy"]
print("policy decision:", decision, "| config policy:", cfg["determinism"]["policy"], "| train.steps:", cfg["train"]["steps"],
      "| delta:", cfg["analysis"]["delta_mape_points"], "| k:", cfg["analysis"]["k"])
assert decision == cfg["determinism"]["policy"] == "P1" and cfg["train"]["steps"] == 8000
assert cfg["analysis"]["delta_mape_points"] == 0.10 and cfg["analysis"]["k"] == 2.0
assert "______" not in open("PREREGISTRATION.md").read()
print("main runs already present:", len(glob.glob("results/runs/main/*.json")), "(expected 0 in a fresh session)")

In [ ]:
# 4. The nine runs, then the frozen-rule analysis. Each run prints one line when it finishes (about 5-6 minutes each).
#    Do NOT interrupt this cell. If it stops, paste the whole output; re-running resumes, but that is then a resumed experiment.
import glob, os, subprocess, sys

def status():
    done = sorted(os.path.basename(f) for f in glob.glob("results/runs/main/*attempt*.json"))
    marks = sorted(os.path.basename(f) for f in glob.glob("results/runs/main/*.running"))
    print(f"main attempts recorded: {len(done)} | unfinished markers: {marks or 'none'}")

status()
proc = subprocess.Popen([sys.executable, "-u", "src/run_main.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
status()
print("exit code:", proc.returncode)
assert proc.returncode == 0 and os.path.exists("results/main_summary.json"), "The main experiment did not finish. Paste this cell's whole output."

In [ ]:
# 5. Package the outputs (kept apart from smoke, calibration and pilot results) and download them.
import glob, os, subprocess
paths = ["results/runs/main", "results/main_summary.json", "results/main_runs.csv", "results/paired_differences.csv",
         "results/env_report.json", "results/session_manifest.json"] + glob.glob("results/pip_freeze_*.txt")
paths = [p for p in paths if os.path.exists(p)]
subprocess.run(["zip", "-rq", "week7_main_outputs.zip", *paths], check=True)
print("zipped:", paths)
from google.colab import files
files.download("week7_main_outputs.zip")

**Stop here.** Send back `week7_main_outputs.zip`. The nine runs are reviewed first. The final interpretation, any decision about the optional extension, and the report all come after that review.